# Reading and Writing Data with DataFrames

Databricks free utilizado => https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/4051382589707593?o=7474657872577658

# 2.3 Demo - Reading and Writing Data with DataFrames

In this demonstration we play the role of a data team onboarding a company's **customer** dataset. We will read the raw files into a DataFrame, inspect and fix its schema, save it as a governed **Delta** table in Unity Catalog, query it with SQL, and handle a column of semi-structured JSON with the Spark 4 `VARIANT` type.

---

## Learning Objectives

By the end of this demo, you will be able to:

* Read CSV files into a DataFrame with inferred/explicit schemas.
* Conform the data to a schema.
* Persist the DataFrame to a Delta table in Unity Catalog.

---

## Table of Contents / Outline

### A. Classroom Setup
* Classroom Setup
* Capture the Catalog Name

### B. Reading Data
* **B1. Schema Inference** — Read CSV with Schema Inference
* **B2. When Inference Guesses Wrong** — Inspect the Inferred Schema
* **B3. Explicitly Defining a Schema** — Read CSV with an Explicit Schema
* **B4. The Same Schema as DDL String** — Read CSV with a DDL Schema
* **B5. Comparing the Three Approaches** — Compare Inferred vs Explicit vs DDL
* **B6. Previewing Data with `display()`** — Preview the Data

### C. Writing Data from DataFrames
* **C1. Writing to the File System** — Write the DataFrame / List the Output Files
* **C2. Writing to a Delta Table** — Save the DataFrame

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the `0 - Required Setup` notebook has been run, sets your default catalog, and sets the default schema to `intro_spark`.

```python
%run ../Includes/Classroom-Setup-2

In [0]:
# Capture the Catalog Name
catalog = spark.catalog.currentCatalog()
source_path = f"/Volumes/{catalog}/intro_spark/source_files/customers"
# example => databricks
print(f"Reading source files from: {source_path}")

Reading source files from: /Volumes/workspace/intro_spark/source_files/customers


## B. Reading Data

Our source is a directory of CSV files describing customers. We will read it three ways — inferring the schema, defining it with a StructType, and defining it with a DDL string — and then compare what each approach gives us.

# Read CSV with Schema Inference

Let's start by asking Spark to infer the schema from the data itself.

In [0]:
# Read CSV with Schema Inference
customers_inferred_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

customers_inferred_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- nation_key: integer (nullable = true)
 |-- phone: long (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- comment: string (nullable = true)



In [0]:
customers_inferred_df.show(5)

+-----------+--------------+------------+----------+----------+---------------+--------------+--------------------+
|customer_id|          name|     address|nation_key|     phone|account_balance|market_segment|             comment|
+-----------+--------------+------------+----------+----------+---------------+--------------+--------------------+
|          1|    John Smith| 123 Main St|         1| 123456789|         1500.5|      Building|Regular customer ...|
|          2|      Jane Doe| 456 Oak Ave|         2| 987654321|        3200.75|     Machinery|Prefers email con...|
|          3|   Bob Johnson| 789 Pine Rd|         3|5551234567|          750.0|     Furniture|     Account on hold|
|          4|Alice Williams|  321 Elm St|         1| 112233445|        5400.25|      Building|        VIP customer|
|          5| Charlie Brown|654 Maple Dr|         4| 998877665|            0.0|     Machinery|         New account|
+-----------+--------------+------------+----------+----------+---------

> ⚠️ **Reading is lazy - so why did that take a moment?**
>
> Most DataFrame reads are lazy, but `inferSchema` is the exception: to guess the column types, Spark has to actually scan the files first. That is a real Spark job - an extra pass over all the data before you have even asked a question of it. On a large dataset that cost is significant.

> 📝 **Headers and Column Names**
>
> If the dataset has a header row, column names are inferred from it. Without a header, columns are named `_c0`, `_c1`, and so on, and must be renamed manually.

---

## B2. When Inference Guesses Wrong

Inference is convenient, but it makes its best guess from the characters it sees - and it can guess wrong. The `phone` column is the classic example: it looks numeric, so let's see how Spark typed it.

In [0]:
# Rename the 'id' column to 'customer_id'
customers_inferred_df = customers_inferred_df.withColumnRenamed("id", "customer_id")

customers_inferred_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- nation_key: integer (nullable = true)
 |-- phone: long (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- comment: string (nullable = true)



In [0]:
# Inspect the Inferred Phone Column
# What type did inference choose for phone, and what do the values look like?
phone_type = dict(customers_inferred_df.dtypes)["phone"]
print(f"Inference typed 'phone' as: {phone_type}\n")

display(customers_inferred_df.select("customer_id", "phone").limit(5))

Inference typed 'phone' as: bigint



customer_id,phone
1,123456789
2,987654321
3,5551234567
4,112233445
5,998877665


> ⚠️ **Why This Matters**
>
> A phone number (or a ZIP/postal code, or an account number) is an *identifier*, not a quantity - you never do arithmetic on it. If inference stores it as a number, any leading zeros are lost and formatting characters may be dropped. The fix is to take control of the schema, which is exactly what we do next.

---

## B3. Explicitly Defining a Schema with StructType

Let's load the same dataset again, this time declaring the schema ourselves with `StructType` and `StructField` objects. This skips the extra inference pass and guarantees the types we want - notably `phone` as a `StringType`.

In [0]:
# Read CSV with an Explicit StructType Schema
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Define explicit schema using StructType. Note phone is a StringType on purpose.
customer_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("nation_key", LongType(), True),
    StructField("phone", StringType(), True),
    StructField("account_balance", DoubleType(), True),
    StructField("market_segment", StringType(), True),
    StructField("comment", StringType(), True)
])

# Read the customers CSV files with the explicit schema
customers_structtype_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load(source_path)
)

In [0]:
customers_inferred_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- nation_key: integer (nullable = true)
 |-- phone: long (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- comment: string (nullable = true)



In [0]:
customers_structtype_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- nation_key: long (nullable = true)
 |-- phone: string (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- comment: string (nullable = true)



In [0]:
customers_structtype_df.show(5) # example

+-----------+--------------+------------+----------+-----------+---------------+--------------+--------------------+
|customer_id|          name|     address|nation_key|      phone|account_balance|market_segment|             comment|
+-----------+--------------+------------+----------+-----------+---------------+--------------+--------------------+
|          1|    John Smith| 123 Main St|         1| 0123456789|         1500.5|      Building|Regular customer ...|
|          2|      Jane Doe| 456 Oak Ave|         2| 0987654321|        3200.75|     Machinery|Prefers email con...|
|          3|   Bob Johnson| 789 Pine Rd|         3|05551234567|          750.0|     Furniture|     Account on hold|
|          4|Alice Williams|  321 Elm St|         1| 0112233445|        5400.25|      Building|        VIP customer|
|          5| Charlie Brown|654 Maple Dr|         4| 0998877665|            0.0|     Machinery|         New account|
+-----------+--------------+------------+----------+-----------+

In [0]:
customers_inferred_df.show(5)

+-----------+--------------+------------+----------+----------+---------------+--------------+--------------------+
|customer_id|          name|     address|nation_key|     phone|account_balance|market_segment|             comment|
+-----------+--------------+------------+----------+----------+---------------+--------------+--------------------+
|          1|    John Smith| 123 Main St|         1| 123456789|         1500.5|      Building|Regular customer ...|
|          2|      Jane Doe| 456 Oak Ave|         2| 987654321|        3200.75|     Machinery|Prefers email con...|
|          3|   Bob Johnson| 789 Pine Rd|         3|5551234567|          750.0|     Furniture|     Account on hold|
|          4|Alice Williams|  321 Elm St|         1| 112233445|        5400.25|      Building|        VIP customer|
|          5| Charlie Brown|654 Maple Dr|         4| 998877665|            0.0|     Machinery|         New account|
+-----------+--------------+------------+----------+----------+---------

## B4. The Same Schema as a DDL String

A `StructType` is precise but verbose. A **DDL string** expresses the exact same schema in a form that is much easier to read and write - and it is the style you will see most often in Databricks.

In [0]:
source_path

'/Volumes/workspace/intro_spark/source_files/customers'

In [0]:
# Read CSV with a DDL Schema
ddl_schema = """
    customer_id BIGINT NOT NULL,
    name STRING,
    address STRING,
    nation_key BIGINT,
    phone STRING,
    account_balance DOUBLE,
    market_segment STRING,
    comment STRING
"""

customers_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(ddl_schema)
    .load(source_path)
)

customers_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- nation_key: long (nullable = true)
 |-- phone: string (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- comment: string (nullable = true)



In [0]:
customers_df.show(5)

+-----------+--------------+------------+----------+-----------+---------------+--------------+--------------------+
|customer_id|          name|     address|nation_key|      phone|account_balance|market_segment|             comment|
+-----------+--------------+------------+----------+-----------+---------------+--------------+--------------------+
|          1|    John Smith| 123 Main St|         1| 0123456789|         1500.5|      Building|Regular customer ...|
|          2|      Jane Doe| 456 Oak Ave|         2| 0987654321|        3200.75|     Machinery|Prefers email con...|
|          3|   Bob Johnson| 789 Pine Rd|         3|05551234567|          750.0|     Furniture|     Account on hold|
|          4|Alice Williams|  321 Elm St|         1| 0112233445|        5400.25|      Building|        VIP customer|
|          5| Charlie Brown|654 Maple Dr|         4| 0998877665|            0.0|     Machinery|         New account|
+-----------+--------------+------------+----------+-----------+

## B5. Comparing the Three Approaches

All three DataFrames read the same files, but they are not equivalent. Run the cell below to see the difference at a glance.

In [0]:
# Compare Inferred vs Explicit Schemas
# The StructType and DDL DataFrames are identical; the inferred one differs on phone.
structtype_matches_ddl = customers_structtype_df.schema == customers_df.schema
inferred_phone = dict(customers_inferred_df.dtypes)["phone"]
explicit_phone = dict(customers_df.dtypes)["phone"]

print(f"StructType schema == DDL schema? {structtype_matches_ddl}")
print(f"'phone' type - Inferred: {inferred_phone} | Explicit: {explicit_phone}")
print(f"\nTakeaway: Inference costs an extra scan AND can pick the wrong type.")
print("Explicit schemas (StructType or DDL) are faster and predictable - prefer them.")

StructType schema == DDL schema? True
'phone' type - Inferred: bigint | Explicit: string

Takeaway: Inference costs an extra scan AND can pick the wrong type.
Explicit schemas (StructType or DDL) are faster and predictable - prefer them.


## B6. Previewing Data with display()

The `display()` function is available in Databricks notebooks. It triggers a Spark action and renders results in an interactive table. As it is intended for visual inspection, results are limited to 10,000 rows or 2 MB, whichever is less. It also lets you:

* Sort and filter the result set
* Download results to CSV
* Build a chart with **+ > Visualization**
* Profile the data with **+ > Data Profile**

In [0]:
# Preview the Data with display()
display(customers_df)

customer_id,name,address,nation_key,phone,account_balance,market_segment,comment
1,John Smith,123 Main St,1,0123456789,1500.5,Building,Regular customer since 2020
2,Jane Doe,456 Oak Ave,2,0987654321,3200.75,Machinery,Prefers email contact
3,Bob Johnson,789 Pine Rd,3,05551234567,750.0,Furniture,Account on hold
4,Alice Williams,321 Elm St,1,0112233445,5400.25,Building,VIP customer
5,Charlie Brown,654 Maple Dr,4,0998877665,0.0,Machinery,New account
7,Eve Adams,147 Birch Way,1,0332211098,890.5,Building,Seasonal buyer
8,Frank Wilson,258 Spruce Ct,3,0223344556,4300.1,Machinery,Bulk orders
9,Grace Lee,369 Walnut St,2,0119988776,6700.8,Furniture,Long-term partner
10,Henry Taylor,741 Aspen Rd,4,0554477889,1200.0,Building,Review account terms


Databricks visualization. Run in Databricks to view.

![image_1791842084393.png](./image_1791842084393.png "image_1791842084393.png")

In [0]:
#Clicar em visualização

> 💡 **Try It: Data Profile**
>
> In the result table above, click **+** and choose **Data Profile**. Spark computes summary statistics across the whole DataFrame. **Question to answer:** what is the average `account_balance`, and which `market_segment` appears most often?

---

## C. Writing Data from DataFrames

Now let's persist our clean `customers_df` (the DDL-schema DataFrame). We will write it two ways: as raw Parquet files, and as a governed Delta table in Unity Catalog.


## C1. Writing to the File System (Parquet)

First we write Parquet files to the output volume created during setup, using the `write` and `save` methods.

In [0]:
# Write the DataFrame as Parquet Files
parquet_output_path = f"/Volumes/{catalog}/intro_spark/output/customers_parquet"

(customers_df.write
    .format("parquet")
    .mode("overwrite")
    .save(parquet_output_path)
)

In [0]:
# List the Output Files
display(dbutils.fs.ls(parquet_output_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/intro_spark/output/customers_parquet/_SUCCESS,_SUCCESS,0,1791845795000
dbfs:/Volumes/workspace/intro_spark/output/customers_parquet/_committed_7287250566361519599,_committed_7287250566361519599,224,1791845794000
dbfs:/Volumes/workspace/intro_spark/output/customers_parquet/_started_7287250566361519599,_started_7287250566361519599,0,1791845794000
dbfs:/Volumes/workspace/intro_spark/output/customers_parquet/part-00000-tid-7287250566361519599-05f3a7b4-1cf8-4314-8f20-c1e56899d697-152-1.c000.snappy.parquet,part-00000-tid-7287250566361519599-05f3a7b4-1cf8-4314-8f20-c1e56899d697-152-1.c000.snappy.parquet,2863,1791845794000
dbfs:/Volumes/workspace/intro_spark/output/customers_parquet/part-00001-tid-7287250566361519599-05f3a7b4-1cf8-4314-8f20-c1e56899d697-153-1.c000.snappy.parquet,part-00001-tid-7287250566361519599-05f3a7b4-1cf8-4314-8f20-c1e56899d697-153-1.c000.snappy.parquet,2770,1791845794000


> 🔍 **What to Observe**
>
> The output is a directory of Parquet part files, not a single file - one file per partition, written in parallel by the Executors.
>
> Parquet is a columnar, self-describing format: the schema travels with the files, so reading them back requires no inference and no DDL.

---

## C2. Writing to a Delta Table in Unity Catalog

`saveAsTable` registers the data as a managed table in Unity Catalog. On Databricks the **default table format is Delta** - so even though we never say "delta", that is what we get: Parquet data files plus a transaction log that adds ACID transactions, time travel, and schema enforcement.

In [0]:
# Save the DataFrame as a Managed Table (this is an action)
customers_df.write.mode("overwrite").saveAsTable("customers")

In [0]:
# Confirm the Table Format is Delta
# DESCRIBE DETAIL reports the physical format of the table we just created.
display(spark.sql("DESCRIBE DETAIL customers").select("format", "name", "numFiles", "location"))

format,name,numFiles,location
delta,workspace.default.customers,1,


Quando executas o comando saveAsTable("customers") sem especificar uma diretoria ou caminho absoluto (como fazemos com ficheiros Parquet), o Databricks guarda a tabela como uma tabela gerida (managed table) no Unity Catalog

Catálogo: O catálogo ativo atual (por exemplo, o teu catálogo pessoal ou de laboratório, obtido por spark.catalog.currentCatalog()).

In [0]:
spark.catalog.currentCatalog()

'workspace'

In [0]:
display(spark.sql("DESCRIBE DETAIL customers").select("location"))

location
""


> 💡 **Delta = Parquet + a Transaction Log**
>
> The format above is `delta`. Delta stores your data as Parquet but adds a transaction log (`_delta_log`) that brings ACID guarantees, schema enforcement, and versioned history. It is the default for Databricks tables and the foundation of the Lakehouse.

---

## C3. Alternative: the DataFrameWriterV2 API

### C3. Alternative: the DataFrameWriterV2 API

`saveAsTable` uses the classic writer. Spark also offers the newer, more explicit `writeTo` builder (`DataFrameWriterV2`). Here we write to a separate table so you can compare the two rather than overwrite the first one.

In [0]:
# Save Using writeTo (DataFrameWriterV2)
# createOrReplace() is explicit about intent; writeTo also supports .append(),
# .overwritePartitions(), and .partitionBy() for finer control.
(customers_df.writeTo(f"{catalog}.intro_spark.customers_v2")
    .createOrReplace()
)

print("Wrote table customers_v2 using the DataFrameWriterV2 API.")

Wrote table customers_v2 using the DataFrameWriterV2 API.


## D. Querying with Spark SQL

The DataFrame API and SQL are two interfaces to the same engine. There are two ways to reach our data from SQL:

* Query the table we saved (`customers`) directly - it lives in Unity Catalog
* Register any DataFrame as a temporary view and query that

Let's do both.

In [0]:
%sql
-- Query the Delta Table with SQL
SELECT market_segment,
       COUNT(*)          AS num_customers,
       ROUND(AVG(account_balance), 2) AS avg_balance
FROM customers
GROUP BY market_segment
ORDER BY num_customers DESC;

market_segment,num_customers,avg_balance
Building,4,2247.81
Machinery,3,2500.28
Furniture,2,3725.4


In [0]:
%sql
-- Query the Delta Table with SQL
SELECT market_segment,
       COUNT(*)          AS num_customers,
       ROUND(AVG(account_balance), 2) AS avg_balance
FROM workspace.default.customers
GROUP BY market_segment
ORDER BY num_customers DESC;

market_segment,num_customers,avg_balance
Building,4,2247.81
Machinery,3,2500.28
Furniture,2,3725.4


In [0]:
# Register a DataFrame as a Temp View
# This bridges the DataFrame API and SQL without persisting a table.
customers_df.createOrReplaceTempView("customers_view")

# spark.sql() runs SQL from Python and returns a DataFrame, so the two compose freely
top_balances = spark.sql("""
    SELECT customer_id, name, market_segment, account_balance
    FROM customers_view
    ORDER BY account_balance DESC
    LIMIT 10
""")

display(top_balances)

customer_id,name,market_segment,account_balance
9,Grace Lee,Furniture,6700.8
4,Alice Williams,Building,5400.25
8,Frank Wilson,Machinery,4300.1
2,Jane Doe,Machinery,3200.75
1,John Smith,Building,1500.5
10,Henry Taylor,Building,1200.0
7,Eve Adams,Building,890.5
3,Bob Johnson,Furniture,750.0
5,Charlie Brown,Machinery,0.0


In [0]:
customers_df.show()

+-----------+--------------+-------------+----------+-----------+---------------+--------------+--------------------+
|customer_id|          name|      address|nation_key|      phone|account_balance|market_segment|             comment|
+-----------+--------------+-------------+----------+-----------+---------------+--------------+--------------------+
|          1|    John Smith|  123 Main St|         1| 0123456789|         1500.5|      Building|Regular customer ...|
|          2|      Jane Doe|  456 Oak Ave|         2| 0987654321|        3200.75|     Machinery|Prefers email con...|
|          3|   Bob Johnson|  789 Pine Rd|         3|05551234567|          750.0|     Furniture|     Account on hold|
|          4|Alice Williams|   321 Elm St|         1| 0112233445|        5400.25|      Building|        VIP customer|
|          5| Charlie Brown| 654 Maple Dr|         4| 0998877665|            0.0|     Machinery|         New account|
|          7|     Eve Adams|147 Birch Way|         1| 03

In [0]:
# Mostra todas as tabelas dentro do esquema intro_spark
display(spark.sql("SHOW TABLES IN intro_spark"))

database,tableName,isTemporary
intro_spark,customers_v2,false
,customers_view,true


In [0]:
current_catalog = spark.catalog.currentCatalog()
current_database = spark.catalog.currentDatabase()

print(f"Catálogo atual: {current_catalog}")
print(f"Esquema atual: {current_database}")

Catálogo atual: workspace
Esquema atual: default


In [0]:
# Conta o número total de linhas na tabela customers
row_count = spark.table("customers").count()
print(f"Total de registos na tabela customers: {row_count}")

Total de registos na tabela customers: 9


## E. Semi-Structured Data with VARIANT (Spark 4)

Real customer feeds often carry a blob of JSON - preferences, tags, device info. Before Spark 4 you stored that as a `STRING` and re-parsed it on every query. Spark 4's `VARIANT` type stores the parsed structure in an optimized binary form, so you can navigate it with simple path expressions and it stays fast.

In [0]:
# Build a Small DataFrame with a JSON Column
import pyspark.sql.functions as F

raw = spark.createDataFrame([
    (1, '{"tier": "gold", "region": "EU", "tags": ["vip", "early_adopter"]}'),
    (2, '{"tier": "silver", "region": "US", "tags": ["newsletter"]}'),
    (3, '{"tier": "platinum", "region": "APAC", "tags": ["vip", "beta"]}')
], "customer_id BIGINT, profile_json STRING")

# Parse the JSON string into a VARIANT column
profiles_df = raw.withColumn("profile", F.parse_json("profile_json")).drop("profile_json")
profiles_df.printSchema()
display(profiles_df)

root
 |-- customer_id: long (nullable = true)
 |-- profile: variant (nullable = true)



customer_id,profile
1,"{""region"":""EU"",""tags"":[""vip"",""early_adopter""],""tier"":""gold""}"
2,"{""region"":""US"",""tags"":[""newsletter""],""tier"":""silver""}"
3,"{""region"":""APAC"",""tags"":[""vip"",""beta""],""tier"":""platinum""}"


In [0]:
# Query Inside the VARIANT with Path Expressions
profiles_df.createOrReplaceTempView("profiles_view")

display(spark.sql("""
    SELECT customer_id,
           profile:tier::string    AS tier,
           profile:region::string  AS region,
           profile:tags[0]::string AS first_tag
    FROM profiles_view
    ORDER BY customer_id
"""))

customer_id,tier,region,first_tag
1,gold,EU,vip
2,silver,US,newsletter
3,platinum,APAC,vip


In [0]:
# Mostra todas as tabelas dentro do esquema intro_spark
display(spark.sql("SHOW TABLES IN intro_spark"))

database,tableName,isTemporary
intro_spark,customers_v2,false
,customers_view,true
,profiles_view,true


In [0]:
display(spark.sql("""
    SELECT customer_id,
           profile:tier::string    AS tier,
           profile:region::string  AS region,
           profile:tags[0]::string AS first_tag,
           profile:tags[1]::string AS second_tag
    FROM profiles_view
    ORDER BY customer_id
"""))

customer_id,tier,region,first_tag,second_tag
1,gold,EU,vip,early_adopter
2,silver,US,newsletter,null
3,platinum,APAC,vip,beta


> 💡 **The `::` Path Syntax**
>
> Notice how we reach into the JSON with `profile:tier`, index an array with `profile:tags[0]`, and cast the extracted value with `::string`. No `from_json`, no pre-declared struct - the shape can vary from row to row and `VARIANT` handles it.

---

## F. Bonus: Delta Time Travel

Because `customers` is a Delta table, every write is a versioned commit. `DESCRIBE HISTORY` shows the log, and you can query any past version - a capability you get by default because Delta is the table format.

In [0]:
%sql
-- View the Table History
DESCRIBE HISTORY customers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-12T22:56:50.000Z,79053712791711,fabieneaulas@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4051382589707593),8940c8b1-fad0-4f5e-aade-8860a1498be9,1012-225106-wtqsjjug-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3133, numDeletionVectorsRemoved -> 0, numOutputRows -> 9, numOutputBytes -> 3133)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-12T22:56:44.000Z,79053712791711,fabieneaulas@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4051382589707593),808f6d64-1a0d-4633-9e25-36675749a981,1012-225106-wtqsjjug-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 9, numOutputBytes -> 3133)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


## Summary

### Reading data into DataFrames:
* Schema inference is convenient but costs an extra scan of the data and can pick the wrong type (we saw `phone`).
* Prefer an explicit schema, defined with `StructType` objects or a readable DDL string - both produce the same result.

### Writing data from DataFrames:
* `df.write.save(...)` writes files (such as Parquet) to a volume; the output is a directory of part files.
* `saveAsTable` and `writeTo` register tables in Unity Catalog, and the default format is **Delta** (Parquet + a transaction log).

### Querying and semi-structured data:
* A temporary view bridges the DataFrame API and SQL - they are one engine.
* Delta gives you versioned history and time travel via `DESCRIBE HISTORY`.
* The Spark 4 `VARIANT` type stores JSON in optimized binary form and is queried with `col:path` expressions.

---

## Knowledge Check